# DPFunc single-protein preprocessing (fixed)
This notebook prepares **one protein** for DPFunc prediction with the **official DPFunc_model checkpoints**.

Outputs (for `ONTOLOGY=mf`):
- `processed_file/mf_test_used_pid_list.pkl`
- `processed_file/pdb_points.pkl`
- `processed_file/pdb_seqs.pkl`
- `processed_file/esm_emds/esm_part_0.pkl` (**ESM2 t33 650M, 1280-dim**)
- `processed_file/graph_features/mf_test_whole_pdb_part0.pkl`
- `processed_file/mf_test_pid_go.txt` (dummy GO label so DPFunc code doesn’t crash)
- `processed_file/mf_test_interpro_file.pkl` (**scipy.sparse.csr_matrix**, shape `(1, 22369)`, zeros)

Then run:
```bash
python DPFunc_pred.py -d mf -n 0 -p DPFunc_model
```

Notes:
- InterPro is all-zeros (so you can run without InterProScan).
- InterPro feature size must match the checkpoint (**22369**).


In [1]:
import os
import pickle as pkl
from pathlib import Path
import numpy as np


## 0) Set inputs

In [ ]:
# EDIT THESE
PID = "A0S864"
ONTOLOGY = "mf"  # "mf", "bp", "cc"
PDB_PATH = f"./data/PDB/{PID}.pdb"

# Constants from DPFunc_model checkpoints
INTERPRO_DIM = 22369


## 1) Create folder structure + pid list

In [ ]:
def save_pkl(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pkl.dump(obj, f)

Path("./processed_file/graph_features").mkdir(parents=True, exist_ok=True)
Path("./processed_file/esm_emds").mkdir(parents=True, exist_ok=True)
Path("./processed_file/interpro").mkdir(parents=True, exist_ok=True)
Path("./results").mkdir(parents=True, exist_ok=True)
Path("./configure").mkdir(parents=True, exist_ok=True)

save_pkl(f"./processed_file/{ONTOLOGY}_test_used_pid_list.pkl", [PID])
print("Wrote:", f"./processed_file/{ONTOLOGY}_test_used_pid_list.pkl")


## 2) Extract CA coordinates + sequence from the PDB

In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

def extract_sequence_and_ca_coords(pdb_file, chain_id=None):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    model = structure[0]

    if chain_id is None:
        chains = list(model.get_chains())
        if not chains:
            raise ValueError("No chains found in PDB")
        chain = chains[0]
    else:
        chain = model[chain_id]

    seq = []
    ca_coords = []
    for residue in chain:
        if residue.id[0] != " ":
            continue
        if "CA" not in residue:
            continue
        try:
            aa = seq1(residue.resname)
        except Exception:
            continue
        seq.append(aa)
        ca_coords.append(residue["CA"].coord.astype(float))

    if not seq or not ca_coords:
        raise ValueError("Failed to extract sequence/CA coords (check chain selection and PDB completeness).")

    return "".join(seq), np.vstack(ca_coords)

seq, coords = extract_sequence_and_ca_coords(PDB_PATH, chain_id=None)
print("PID:", PID)
print("Sequence length:", len(seq))
print("CA coords shape:", coords.shape)

save_pkl("./processed_file/pdb_points.pkl", {PID: coords})
save_pkl("./processed_file/pdb_seqs.pkl", {PID: seq})
print("Wrote processed_file/pdb_points.pkl and pdb_seqs.pkl")


## 3) Residue embeddings (ESM2 t33 650M, 1280-dim)

In [ ]:
import torch
import esm

def embed_esm2_t33_650M(seq: str) -> np.ndarray:
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    model.eval()
    batch_converter = alphabet.get_batch_converter()
    _, _, toks = batch_converter([("protein", seq)])
    with torch.no_grad():
        out = model(toks, repr_layers=[model.num_layers], return_contacts=False)
    rep = out["representations"][model.num_layers][0, 1:1+len(seq)]  # [L, 1280]
    return rep.cpu().numpy().astype(np.float32)

emb = embed_esm2_t33_650M(seq)
assert emb.shape[0] == len(seq)
assert emb.shape[1] == 1280, emb.shape

save_pkl("./processed_file/esm_emds/esm_part_0.pkl", {PID: emb})
print("Wrote processed_file/esm_emds/esm_part_0.pkl with shape", emb.shape)


## 4) Build the DGL graph and save graph_features pickle

In [ ]:
import dgl
import torch

def build_graph_from_points(points: np.ndarray, threshold: float = 12.0):
    L = points.shape[0]
    u, v, dis = [], [], []
    for i in range(L):
        pi = points[i]
        for j in range(L):
            if i == j:
                continue
            d = float(np.linalg.norm(pi - points[j]))
            if d <= threshold:
                u.append(i); v.append(j); dis.append(d)

    g = dgl.graph((torch.tensor(u), torch.tensor(v)), num_nodes=L)
    g.edata["dis"] = torch.tensor(dis, dtype=torch.float32)
    return g

g = build_graph_from_points(coords, threshold=12.0)
g.ndata["x"] = torch.from_numpy(emb)

out_graph_path = f"./processed_file/graph_features/{ONTOLOGY}_test_whole_pdb_part0.pkl"
save_pkl(out_graph_path, [g])
print("Wrote:", out_graph_path, "nodes:", g.num_nodes(), "featdim:", g.ndata["x"].shape[1])


## 5) Dummy GO label file (required by DPFunc code)

In [ ]:
dummy_go = "GO:0003674"  # molecular_function
go_path = f"./processed_file/{ONTOLOGY}_test_pid_go.txt"
with open(go_path, "w") as f:
    f.write(f"{PID}\t{dummy_go}\n")
print("Wrote:", go_path)


## 6) InterPro features (csr_matrix zeros, required format)

In [ ]:
import scipy.sparse as sp

X = sp.csr_matrix((1, INTERPRO_DIM), dtype=np.float32)

with open(f"./processed_file/{ONTOLOGY}_test_interpro_file.pkl", "wb") as f:
    pkl.dump(X, f)

save_pkl(f"./processed_file/interpro/{PID}.pkl", np.zeros(INTERPRO_DIM, dtype=np.float32))

print("Wrote:", f"./processed_file/{ONTOLOGY}_test_interpro_file.pkl", "csr_matrix", X.shape)
print("Wrote:", f"./processed_file/interpro/{PID}.pkl", "len", INTERPRO_DIM)


## 7) Write configure/<ont>.yaml to point DPFunc_pred.py at these files

In [ ]:
cfg_path = f"./configure/{ONTOLOGY}.yaml"
yaml_lines = [
    f"name: {ONTOLOGY}\n",
    f"mlb: ./mlb/{ONTOLOGY}_go.mlb\n",
    "results: ./results\n\n",
    "base:\n",
    "  interpro_whole: ./processed_file/interpro/{}.pkl\n\n",
    "test:\n",
    "  name: test\n",
    f"  pid_list_file: ./processed_file/{ONTOLOGY}_test_used_pid_list.pkl\n",
    f"  pid_pdb_file: ./processed_file/graph_features/{ONTOLOGY}_test_whole_pdb_part0.pkl\n",
    f"  pid_go_file: ./processed_file/{ONTOLOGY}_test_pid_go.txt\n",
    f"  interpro_file: ./processed_file/{ONTOLOGY}_test_interpro_file.pkl\n",
]
with open(cfg_path, "w") as f:
    f.writelines(yaml_lines)
print("Wrote:", cfg_path)


## 8) Run prediction

In [ ]:
print(f"Run:\n  python DPFunc_pred.py -d {ONTOLOGY} -n 0 -p DPFunc_model")
print("Predictions will be in ./results/DPFunc_model_<ont>_final.pkl")


In [8]:
from pathlib import Path

Path(__file__)

NameError: name '__file__' is not defined

In [1]:
from DPFunctional import get_GO_terms

PID = "A0S864"
PDB_PATH = "./data/PDB/A0S864.pdb"
results = get_GO_terms(PDB_PATH, PID, debug=True)

🔗 Extracting sequence and Cα coords from PDB
🔢 Creating the ESM embedding
🕸️ Building the DGL graph
🚀 Runing DPFunc
✅ Done. Results...
  protein_id           gos                                        predictions
0     A0S864  {GO:0003674}  {'GO:0000006': 0.00014232037938199937, 'GO:000...


In [6]:
results.predictions

0    {'GO:0000006': 0.00014232037938199937, 'GO:000...
Name: predictions, dtype: object

In [7]:
len(results.predictions.iloc[0])

6086

In [8]:
pred = results.predictions.iloc[0]

top = sorted(pred.items(), key=lambda x: x[1], reverse=True)[:10]
top

[('GO:0003674', 0.99965500831604),
 ('GO:0005488', 0.5451997915903727),
 ('GO:0005515', 0.27757759888966876),
 ('GO:0030234', 0.2545125087102254),
 ('GO:0097159', 0.23582683006922403),
 ('GO:0098772', 0.23112879196802774),
 ('GO:1901363', 0.22971109549204508),
 ('GO:0003676', 0.21321765581766763),
 ('GO:0003824', 0.15084302425384521),
 ('GO:0042802', 0.14691853523254395)]